In [ ]:
import pandas as pd
from IPython.display import display
from src.research_config import ResearchConfig
from src.backtest import entry_option_terms, budgeted_size


# 06 Option Pricing and Expiry Selection
Convert the Module 04 snapshot into synthetic next-close option terms using the same pricing and sizing functions as the backtest.


In [ ]:
cfg = ResearchConfig()


In [ ]:
snapshot = pd.read_parquet("signal_snapshot.parquet")
test_prices = pd.read_parquet("test_prices.parquet")
volatility = pd.read_parquet("oos_ewma_volatility.parquet")
rates = pd.read_parquet("risk_free_rates.parquet").iloc[:, 0]
display(snapshot)


In [ ]:
rows = []
for row in snapshot.itertuples():
    i = test_prices.index.get_loc(row.signal_date)
    if row.horizon <= 1 or i + row.horizon >= len(test_prices):
        continue

    entry_date = test_prices.index[i + 1]
    expiry_date = test_prices.index[i + int(row.horizon)]
    instruction = {
        "dependent": row.dependent,
        "independent": row.independent,
        "signal_date": row.signal_date,
        "expiry_date": expiry_date,
        "direction": row.direction,
    }
    spots, types, option_prices, deltas = entry_option_terms(
        instruction, entry_date, test_prices, volatility, rates
    )
    budget = cfg.initial_capital * cfg.premium_budget_fraction
    sizing = budgeted_size(
        row.beta,
        spots,
        deltas,
        option_prices,
        budget,
        cfg.max_hedge_error,
        cfg.slippage_bps,
        cfg.commission_per_contract,
    )
    if sizing is None:
        continue

    rows.append({
        "pair": row.pair,
        "signal_date": row.signal_date,
        "entry_date": entry_date,
        "expiry_date": expiry_date,
        "dependent_type": types[0],
        "independent_type": types[1],
        "dependent_price": option_prices[0],
        "independent_price": option_prices[1],
        "illustrative_budget": budget,
        **sizing,
    })

option_preview = pd.DataFrame(rows)
option_preview.to_parquet("option_preview.parquet")
display(option_preview)
